In [ ]:
import cuvis
import time
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import threading
import cv2
%matplotlib widget  

print("Initializing Cuvis")
cuvis.General.init("./settings")
camera_serial_number_str = "YourCameraSerialNumber"

calib = cuvis.SessionFile(F"./factory/{camera_serial_number_str}.cu3c")
acq_cont = cuvis.AcquisitionContext(calib)

while(not acq_cont.ready):
    time.sleep(1)
    print(".", end="")
print("\nCamera connected!")
time.sleep(1)

acq_cont.set_continuous(False)  # Pause video stream for now
acq_cont.operation_mode = cuvis.OperationMode.Internal
acq_cont.integration_time = 100
acq_cont.fps = 10

# Create ProcessingContext
proc_cont = cuvis.ProcessingContext(calib)
proc_cont.processing_mode = cuvis.ProcessingMode.Raw

def process_and_plot(change=None):

    print("Acquiring...")
    acq_cont.set_continuous(True)

    while not acq_cont.has_next_measurement():
        time.sleep(1)
        print(".", end="")
    
    # Get initial measurement
    result = acq_cont.get_next_measurement(100)
    proc_cont.apply(result)

    app_state = {"cube": result.cube}
    
    fig, (ax_img, ax_spec) = plt.subplots(1, 2, figsize=(10, 5))
    plt.tight_layout()
    mid_channel = app_state["cube"].channels // 2
    
    
    channel = app_state["cube"].array[:, :, mid_channel]
    x = cv2.normalize(channel, 0, 10000, cv2.NORM_MINMAX)

    im = ax_img.imshow(x, cmap="gray")
    marker, = ax_img.plot([], [], "r+", markersize=10, markeredgewidth=2)
    ax_img.set_title(f"Channel {mid_channel}")

    (line,) = ax_spec.plot([], [], lw=1.5)
    vline = ax_spec.axvline(app_state["cube"].wavelength[mid_channel], color="r", ls="--", lw=1)
    ax_spec.set_xlabel("Wavelength")
    ax_spec.set_ylabel("Counts")
    ax_spec.set_title("Spectrum")
    ax_spec.grid(True, alpha=0.3)
    selected_pixel = {"x": None, "y": None}

    # --- Channel slider ---
    channel_slider = widgets.IntSlider(
        value=mid_channel,
        min=0,
        max=app_state["cube"].channels - 1,
        step=1,
        description="Channel",
        continuous_update=True,
    )

    def onclick(event):
        cube = app_state["cube"] # Fetch the most recent cube
        if event.inaxes == ax_img and event.xdata is not None and event.ydata is not None:
            print('clicked! in if')
            x, y = int(event.xdata), int(event.ydata)
            if 0 <= x < cube.array.shape[1] and 0 <= y < cube.array.shape[0]:
                print('clicked! in if if')
                selected_pixel["x"], selected_pixel["y"] = x, y
                marker.set_data([x], [y])

                spectrum = np.array(cube.array[y, x, :]).ravel()
                wavelengths = np.array(cube.wavelength).ravel()
                line.set_data(wavelengths, spectrum)

                ax_spec.relim()
                ax_spec.autoscale_view()
                ch = channel_slider.value
                vline.set_xdata([cube.wavelength[ch]])
                ax_spec.set_title(
                    f"Spectrum at (x={x}, y={y}) — value={cube.array[y, x, ch]:.3f}"
                )
                fig.canvas.draw_idle()

    def on_channel_change(change):
        cube = app_state["cube"] # Fetch the most recent cube
        if change["name"] == "value":
            print('channel_change! in if')
            ch = change["new"]
            channel = cube.array[:, :, ch]
            x = cv2.normalize(channel, 0, 10000, cv2.NORM_MINMAX)
            im.set_data(x)
            ax_img.set_title(f"Channel {ch}")
            vline.set_xdata([cube.wavelength[ch]])

            if selected_pixel["x"] is not None:
                print('channel_change! in if if')
                x, y = selected_pixel["x"], selected_pixel["y"]
                marker.set_data([x], [y])
                ax_spec.set_title(
                    f"Spectrum at (x={x}, y={y}) — value={cube.array[y, x, ch]:.3f}"
                )
            else:
                marker.set_data([], [])
            fig.canvas.draw_idle()
    
    fig.canvas.mpl_connect("button_press_event", onclick)
    channel_slider.observe(on_channel_change, names="value")
    display(channel_slider)
    plt.show()

    def live_view_loop():
        while True:
            if acq_cont.has_next_measurement():
                result = acq_cont.get_next_measurement(100)
                proc_cont.apply(result)
                cube = result.cube
                app_state["cube"] = cube

                ch = channel_slider.value
                channel = cube.array[:, :, ch]
                x = cv2.normalize(channel, 0, 10000, cv2.NORM_MINMAX)
                im.set_data(x)
                fig.canvas.draw_idle()

            else:
                time.sleep(0.1)

    # Start the background thread
    update_thread = threading.Thread(target=live_view_loop)
    update_thread.daemon = True
    update_thread.start()

process_and_plot()

Initializing Cuvis
......
Camera connected!


Image(value=b'', format='jpeg', layout="Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', b…

Acquiring...
dict_keys(['settings_rec', 'PANIMAGE_info', 'PANIMAGE', 'IMAGE_info', ''])
dict_keys(['settings_rec', 'PANIMAGE_info', 'PANIMAGE', 'IMAGE_info', ''])
dict_keys(['settings_rec', 'PANIMAGE_info', 'PANIMAGE', 'IMAGE_info', ''])
dict_keys(['settings_rec', 'PANIMAGE_info', 'PANIMAGE', 'IMAGE_info', ''])
dict_keys(['settings_rec', 'PANIMAGE_info', 'PANIMAGE', 'IMAGE_info', ''])
dict_keys(['settings_rec', 'PANIMAGE_info', 'PANIMAGE', 'IMAGE_info', ''])
dict_keys(['settings_rec', 'PANIMAGE_info', 'PANIMAGE', 'IMAGE_info', ''])
dict_keys(['settings_rec', 'PANIMAGE_info', 'PANIMAGE', 'IMAGE_info', ''])
dict_keys(['settings_rec', 'PANIMAGE_info', 'PANIMAGE', 'IMAGE_info', ''])
dict_keys(['settings_rec', 'PANIMAGE_info', 'PANIMAGE', 'IMAGE_info', ''])
dict_keys(['settings_rec', 'PANIMAGE_info', 'PANIMAGE', 'IMAGE_info', ''])
dict_keys(['settings_rec', 'PANIMAGE_info', 'PANIMAGE', 'IMAGE_info', ''])
dict_keys(['settings_rec', 'PANIMAGE_info', 'PANIMAGE', 'IMAGE_info', ''])
dict_keys(['

KeyboardInterrupt: 